# AI-Based Customer Spending Prediction

**Student: Pranjal Jain**

Beginner ML classification project using Python, Pandas, NumPy, Matplotlib, and Scikit-learn. Dataset is synthetic educational data.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay
raw = pd.read_csv('customer_spending_raw.csv')
display(raw.head())

## 1. Inspect data

In [ ]:
print('Shape:',raw.shape); print(raw.dtypes); print('Missing values:\n',raw.isna().sum()); print('Duplicates:',raw.duplicated().sum()); display(raw.describe(include='all'))

## 2. Clean data

In [ ]:
df=raw.drop_duplicates().copy()
for c in ['Age','Annual_Income','Average_Order_Value']:
    df[c]=pd.to_numeric(df[c],errors='coerce').fillna(pd.to_numeric(df[c],errors='coerce').median())
df['Discount_Preference']=df['Discount_Preference'].fillna(df['Discount_Preference'].mode()[0])
df.to_csv('customer_spending_cleaned.csv',index=False)
print('Before:',len(raw),'After:',len(df)); print(df.isna().sum())

## 3. EDA visualizations
Run these chart cells to inspect class counts, age distribution, income/order value, visits by category, and discount-preference mix.

In [ ]:
df['Spending_Category'].value_counts().reindex(['Low','Medium','High']).plot(kind='bar',title='Customer Count by Spending Category'); plt.tight_layout(); plt.show()

In [ ]:
plt.hist(df.Age,bins=12,edgecolor='black'); plt.title('Customer Age Distribution'); plt.xlabel('Age'); plt.ylabel('Customers'); plt.show()

In [ ]:
for c in ['Low','Medium','High']:
 s=df[df.Spending_Category==c]; plt.scatter(s.Annual_Income,s.Average_Order_Value,label=c,alpha=.6)
plt.title('Income vs Average Order Value'); plt.xlabel('Annual Income'); plt.ylabel('Average Order Value'); plt.legend(); plt.show()

In [ ]:
df.groupby('Spending_Category').Monthly_Visits.mean().reindex(['Low','Medium','High']).plot(kind='bar',title='Average Monthly Visits by Category'); plt.show()

In [ ]:
pd.crosstab(df.Discount_Preference,df.Spending_Category,normalize='index').reindex(columns=['Low','Medium','High']).plot(kind='bar',stacked=True,title='Category Mix by Discount Preference'); plt.show()

## 4. Preprocessing and model training

In [ ]:
features=['Age','Annual_Income','Monthly_Visits','Average_Order_Value','Loyalty_Points','Discount_Preference','Gender']; num=features[:5]; cat=features[5:]
X=df[features]; y=df.Spending_Category
try: oh=OneHotEncoder(handle_unknown='ignore',sparse_output=False)
except TypeError: oh=OneHotEncoder(handle_unknown='ignore',sparse=False)
prep=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',oh)]),cat)])
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
models={'Logistic Regression':Pipeline([('prep',prep),('model',LogisticRegression(max_iter=1500))]),'Decision Tree':Pipeline([('prep',prep),('model',DecisionTreeClassifier(max_depth=5,random_state=42))])}
rows=[]
for name,m in models.items():
 m.fit(Xtr,ytr); p=m.predict(Xte); rows.append({'Model':name,'Accuracy':accuracy_score(yte,p),'Precision macro':precision_score(yte,p,average='macro',zero_division=0),'Recall macro':recall_score(yte,p,average='macro',zero_division=0),'F1 macro':f1_score(yte,p,average='macro',zero_division=0)})
results=pd.DataFrame(rows).set_index('Model'); display(results)

## 5. Confusion matrix and interpretation

In [ ]:
best=results['F1 macro'].idxmax(); model=models[best]
ConfusionMatrixDisplay.from_estimator(model,Xte,yte,labels=['Low','Medium','High']); plt.title('Confusion Matrix — '+best); plt.show()
print('Selected model:',best)

## 6. Sample prediction function

In [ ]:
def predict_customer(age, annual_income, monthly_visits, average_order_value, loyalty_points, discount_preference, gender):
 sample=pd.DataFrame([{'Age':age,'Annual_Income':annual_income,'Monthly_Visits':monthly_visits,'Average_Order_Value':average_order_value,'Loyalty_Points':loyalty_points,'Discount_Preference':discount_preference,'Gender':gender}])
 return model.predict(sample)[0]
print(predict_customer(30,60000,8,2200,5,'Medium','Female'))

## 7. Business observations and limitations

- Visit frequency, income, order value, and loyalty points are useful candidate signals to examine.
- Compare precision, recall, and F1 in addition to accuracy.
- The category labels were generated from a synthetic spending proxy, so metrics reflect an artificial task and are not real-world performance.
- Do not use these predictions for real customer decisions without representative data, validation, privacy review, and fairness checks.

## Conclusion
The project demonstrates data cleaning, EDA, preprocessing, two classification models, evaluation, interpretation, and a sample prediction.